In [40]:
# Cell 1: Setup and Path Routing
import sys
import os

# 1. Get the absolute path to the 'src' directory
# (Since the notebook is in /notebooks, we go up one level '..', then into 'src')
src_path = os.path.abspath(os.path.join('..', 'src'))

# 2. Add 'src' to the system path so Python knows where to look
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 3. Enable autoreload so module changes reflect immediately
%load_ext autoreload
%autoreload 2

print(f"Added {src_path} to Python path!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added /home/musel/Documents/github/agentic-rag/src to Python path!


In [41]:
import os
from dotenv import load_dotenv

# Load API keys
load_dotenv()

# Import your newly modularized code
from schema import IntentSkeleton, ConnectionTarget
from prompts import get_intent_extraction_prompt
from graph import build_graph

In [42]:
# Initialize the graph
intent_graph = build_graph()


In [43]:

# Define your test query
query = '''Configure telemetry for our Nokia SR Linux router (version latest) at clab-telemetry-lab-srl-router:57400. I need to monitor the system CPU, system memory, and specifically the mgmt0 interface statistics. Set up a dial-in stream using gRPC with JSON encoding. The reporting interval should be 2 seconds. Our Prometheus collector is listening at clab-telemetry-lab-prometheus:9804.'''

# Run the graph
result = intent_graph.invoke({
    "query": query,
    "skeleton": None
})

# Inspect the structured JSON output
if result.get("skeleton"):
    print(result["skeleton"].model_dump_json(indent=2))
else:
    print("Graph failed to return a skeleton.")

{
  "device_hints": {
    "vendor": "nokia",
    "nos": "sr-linux",
    "version": "latest"
  },
  "target_router": {
    "address": "clab-telemetry-lab-srl-router",
    "port": 57400
  },
  "collector": {
    "address": "clab-telemetry-lab-prometheus",
    "port": 9804
  },
  "telemetry_goal": [
    "system_cpu",
    "system_memory",
    "interfaces"
  ],
  "target_interfaces": [
    "mgmt0"
  ],
  "sample_interval_ms": 2000,
  "encoding": "json",
  "protocol": "grpc",
  "telemetry_mode": "dial-in",
  "path_requirements": [],
  "notes": [],
  "unresolved": [
    "Specific YANG paths for system CPU, system memory, and mgmt0 interface statistics are needed for Nokia SR Linux."
  ]
}


In [44]:
from tools import deploy_telemetry

In [52]:
# Test the tool directly
deploy_telemetry(result["skeleton"])


"Success: Telemetry deployed. Generated config with 3 paths and restarted 'clab-telemetry-lab-gnmic'."